# 06 — Cloud Deployment: Serverless AI on AWS

## Part 1: Theory

### 1.1 Cloud Computing Models

```
You Manage:  Everything ← IaaS (EC2) ← PaaS (Elastic Beanstalk) ← FaaS (Lambda) ← SaaS → Nothing
```

| Model | Example | You Manage | Provider Manages |
|-------|---------|-----------|------------------|
| IaaS | EC2, GCE | OS, runtime, app | Hardware, network |
| PaaS | Beanstalk, App Engine | App code | OS, scaling, infra |
| FaaS | Lambda, Cloud Functions | Function code | Everything else |
| SaaS | Bedrock, OpenAI API | Nothing | Everything |

### 1.2 AWS Lambda Internals

```
Request → API Gateway → Lambda Service → Execution Environment
                                          ├── /var/runtime (Lambda runtime)
                                          ├── /var/task (your code)
                                          ├── /tmp (512MB-10GB ephemeral storage)
                                          └── Memory: 128MB-10GB (CPU scales with memory)
```

**Cold Start**: First request creates new execution environment (~1-5s for Python)
**Warm Start**: Reuses existing environment (~50-100ms)

**Limits**: 15min timeout, 10GB memory, 250MB deployment package (50MB zipped), 6MB response

### 1.3 Deployment Patterns

| Pattern | How | Risk | Rollback |
|---------|-----|------|----------|
| **Blue-Green** | Two identical envs, switch traffic | Low | Instant (switch back) |
| **Canary** | Route 5% → new, 95% → old | Very low | Remove canary |
| **Rolling** | Replace instances one by one | Medium | Slow (roll back each) |
| **A/B** | Route by user segment | Low | Remove routing rule |

### 1.4 Infrastructure as Code

| Tool | Language | State | Best For |
|------|----------|-------|----------|
| CloudFormation | YAML/JSON | AWS-managed | AWS-only, SAM |
| Terraform | HCL | S3/local | Multi-cloud |
| CDK | Python/TS | CloudFormation | Developers who hate YAML |
| SAM | YAML (extended CF) | AWS-managed | Serverless apps |

### 1.5 AI Workload Deployment Options

```
Latency requirement:
├── <100ms → SageMaker Real-time Endpoint (GPU)
├── <3s → Lambda + External LLM API (OpenRouter/Bedrock)
├── <30s → Lambda (heavy processing, 10GB memory)
└── Minutes → Step Functions + Batch
```

---

### Architecture: Serverless RAG
```
┌────────┐    ┌───────────────┐    ┌─────────────────────────────┐
│ Client │───▶│ API Gateway   │───▶│ Lambda                      │
│        │    │ (HTTP API)    │    │ ┌─────────┐  ┌───────────┐  │
└────────┘    │ • Rate limit  │    │ │ FastAPI │──│ RAG Chain │  │
              │ • Auth (JWT)  │    │ │ +Mangum │  │           │  │
              └───────────────┘    │ └─────────┘  └─────┬─────┘  │
                                   └───────────────────│─────────┘
                                                       │
                                          ┌────────────┼────────────┐
                                          ▼            ▼            ▼
                                   ┌──────────┐ ┌──────────┐ ┌──────────┐
                                   │ S3       │ │ Secrets  │ │OpenRouter│
                                   │ (FAISS)  │ │ Manager  │ │ (LLM)   │
                                   └──────────┘ └──────────┘ └──────────┘
```

## Part 2: Implementation

In [ ]:
import json
import time
import numpy as np
import concurrent.futures

### Lambda Handler (FastAPI + Mangum)

In [ ]:
lambda_code = '''
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from mangum import Mangum
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

app = FastAPI(title="RAG API", version="1.0")

class QueryRequest(BaseModel):
    question: str
    top_k: int = 3

class QueryResponse(BaseModel):
    answer: str
    sources: list[str]
    latency_ms: float

# Lazy-load (cached across warm invocations)
_retriever = None
_chain = None

def get_chain():
    global _retriever, _chain
    if _chain is None:
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        vectorstore = FAISS.load_local("/tmp/faiss_index", embeddings,
                                        allow_dangerous_deserialization=True)
        _retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        llm = ChatOpenAI(
            model="meta-llama/llama-3.1-8b-instruct:free",
            openai_api_key=os.environ["OPENROUTER_API_KEY"],
            openai_api_base="https://openrouter.ai/api/v1",
            temperature=0,
        )
        prompt = ChatPromptTemplate.from_template(
            "Answer based on context. If unsure, say so.\\n\\nContext:\\n{context}\\n\\nQuestion: {question}\\nAnswer:"
        )
        _chain = (
            {"context": _retriever | (lambda docs: "\\n".join(d.page_content for d in docs)),
             "question": RunnablePassthrough()}
            | prompt | llm | StrOutputParser()
        )
    return _retriever, _chain

@app.post("/query", response_model=QueryResponse)
async def query(req: QueryRequest):
    import time
    start = time.time()
    retriever, chain = get_chain()
    docs = retriever.invoke(req.question)
    answer = chain.invoke(req.question)
    return QueryResponse(
        answer=answer,
        sources=[d.page_content[:100] for d in docs],
        latency_ms=(time.time() - start) * 1000
    )

@app.get("/health")
async def health():
    return {"status": "healthy", "model": "llama-3.1-8b-free"}

handler = Mangum(app)
'''
print(lambda_code)

### SAM Template

In [ ]:
sam_template = {
    "AWSTemplateFormatVersion": "2010-09-09",
    "Transform": "AWS::Serverless-2016-10-31",
    "Globals": {"Function": {"Timeout": 30, "MemorySize": 1024, "Runtime": "python3.11",
        "Environment": {"Variables": {"OPENROUTER_API_KEY": "{{resolve:secretsmanager:rag-api/openrouter}}"}}}},
    "Resources": {
        "RagFunction": {"Type": "AWS::Serverless::Function", "Properties": {
            "Handler": "app.handler", "CodeUri": "src/",
            "Events": {"Api": {"Type": "HttpApi", "Properties": {"Path": "/{proxy+}", "Method": "ANY"}}},
            "Policies": ["S3ReadPolicy"]}},
        "IndexBucket": {"Type": "AWS::S3::Bucket"}},
    "Outputs": {"ApiUrl": {"Value": {"Fn::Sub": "https://${ServerlessHttpApi}.execute-api.${AWS::Region}.amazonaws.com"}}}
}
print(json.dumps(sam_template, indent=2))

### Cold Start Optimization

In [ ]:
strategies = {
    "Provisioned Concurrency": {"desc": "Keep N instances warm", "cost": "~$15/month per instance", "cold_start": "0ms"},
    "Lazy Loading + /tmp cache": {"desc": "Load FAISS on first request, reuse on warm", "cost": "$0", "cold_start": "~3s first, ~100ms after"},
    "Smaller Package": {"desc": "Lambda Layers for deps, minimize zip", "cost": "$0", "cold_start": "Reduced by 30-50%"},
    "ARM64 (Graviton)": {"desc": "Use arm64 architecture", "cost": "20% cheaper", "cold_start": "~20% faster"},
    "SnapStart": {"desc": "Snapshot initialized state", "cost": "$0", "cold_start": "~200ms (Java only currently)"},
}
for name, s in strategies.items():
    print(f"\n🔧 {name}\n   {s['desc']}\n   Cost: {s['cost']} | Cold start: {s['cold_start']}")

### Load Testing Simulation

In [ ]:
def simulate_request(i):
    is_cold = i < 5 or np.random.random() < 0.08  # first 5 + 8% cold
    latency = np.random.normal(3.0, 0.5) if is_cold else np.random.normal(0.4, 0.1)
    time.sleep(max(0.001, latency / 50))  # scaled for demo
    return {"latency_ms": max(50, latency * 1000), "cold_start": is_cold, "status": 200 if np.random.random() > 0.02 else 500}

def load_test(num_requests=100, concurrency=10):
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as ex:
        results = list(ex.map(simulate_request, range(num_requests)))
    latencies = [r["latency_ms"] for r in results]
    errors = sum(1 for r in results if r["status"] != 200)
    cold = sum(1 for r in results if r["cold_start"])
    print(f"Load Test: {num_requests} requests, {concurrency} concurrent")
    print(f"  P50: {np.percentile(latencies, 50):.0f}ms")
    print(f"  P95: {np.percentile(latencies, 95):.0f}ms")
    print(f"  P99: {np.percentile(latencies, 99):.0f}ms")
    print(f"  Errors: {errors}/{num_requests} ({errors/num_requests:.1%})")
    print(f"  Cold starts: {cold}/{num_requests} ({cold/num_requests:.1%})")

load_test()

## Part 3: Key Takeaways

1. **Serverless** = zero ops, auto-scale, pay-per-use — ideal for variable AI traffic
2. **Cold starts** are the #1 challenge — mitigate with lazy loading + provisioned concurrency
3. **SAM/IaC** makes deployments reproducible and auditable
4. **External LLM APIs** (OpenRouter) avoid packaging large models in Lambda
5. **Load test before launch** — know your P95 latency and error rate
6. **ARM64 + right-sized memory** = 20-40% cost savings

### Next → 07_observability.ipynb